Q𝐮𝐞𝐬𝐭𝐢𝐨𝐧:

You are given a dataset containing server logs with the following fields:

server_id (string): ID of the server

log_date (string): Date of the log in yyyy-MM-dd format

cpu_usage (integer): CPU usage percentage

Your tasks are:

* For each server, calculate the day-over-day CPU usage difference (day_diff).
* Find the day with maximum CPU usage per server.
* Compute the average CPU usage per server (rounded to 2 decimals).
* Produce a final report that includes all the above information, sorted by average CPU usage (descending).

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
data = [
  ("S1", "2024-01-01", 70),
  ("S1", "2024-01-02", 75),
  ("S1", "2024-01-03", 60),
  ("S2", "2024-01-01", 55),
  ("S2", "2024-01-02", 65),
  ("S2", "2024-01-03", 80),
  ("S3", "2024-01-01", 40),
  ("S3", "2024-01-02", 50),
  ("S3", "2024-01-03", 45)
]

columns = ["server_id", "log_date", "cpu_usage"]

df = spark.createDataFrame(data, columns)


In [0]:
df = df.withColumn("log_date",F.col("log_date").cast("date"))
display(df)

In [0]:
windowSpec = Window.partitionBy("server_id").orderBy(F.col("log_date"))
df = df.withColumn("prev_day",F.lag("log_date").over(windowSpec)) \
    .withColumn("prev_cpu",F.lag("cpu_usage").over(windowSpec)) \
    .withColumn("day_diff",F.datediff("log_date","prev_day")) \
    .withColumn("cpu_diff",F.col("cpu_usage")-F.col("prev_cpu"))

avg_df = df.groupBy("server_id").agg(F.avg("cpu_usage").alias("avg_cpu"))

result = df.join(avg_df, on="server_id", how="inner")

display(result)
